In [ ]:
#Google Colab file
!pip install datasets sqlalchemy psycopg2-binary

In [ ]:
import os
from google.colab import userdata

# Pull secrets into env vars so get_hf_token_optional() can find them
# Must have these two variables for the code to work successfully
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [ ]:
#This file is the same as the original news_ingest.py from the backend folder
# The only file changes are marked with a comment
from __future__ import annotations

import os
import json
import logging
from datetime import datetime, timezone, date
from collections import defaultdict, Counter
from typing import Any, Dict, List, Optional, Tuple

from datasets import load_dataset
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert

#different from news_ingest.py file in repo
# sys.path line commented out - not needed + causes errors in Colab
#sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


logger = logging.getLogger("article_ingestor")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s - %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)


def build_db_url() -> str:
    db_url = os.getenv("DATABASE_URL")
    if db_url:
        return db_url
    db_user = os.getenv("PG_USER", "stock_user")
    db_password = os.getenv("PG_PASS", "stock_pass")
    db_host = os.getenv("PG_HOST", "postgres")
    db_port = os.getenv("PG_PORT", "5432")
    db_name = os.getenv("PG_DB", "stock_db")
    return f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"


def get_hf_token_optional() -> Optional[str]:
    token = (os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN") or "").strip()
    return token or None


def parse_extras_any(extra_fields: Any) -> dict:
    if not extra_fields:
        return {}
    if isinstance(extra_fields, dict):
        return extra_fields
    if isinstance(extra_fields, str):
        s = extra_fields.strip()
        if not s or (s[0] not in "{["):
            return {}
        try:
            v = json.loads(s)
            return v if isinstance(v, dict) else {}
        except Exception:
            return {}
    return {}


def extract_title_and_desc(text: str, max_desc_chars: int = 280) -> Tuple[str, str]:
    if not text:
        return "", ""
    parts = text.split("\n\n", 1)
    title = parts[0].strip()
    body = parts[1].strip() if len(parts) > 1 else ""
    desc = body[:max_desc_chars].strip()
    return title, desc


def pick_url(extras: Dict[str, Any]) -> Optional[str]:
    return extras.get("url") or extras.get("web_url") or extras.get("link") or extras.get("permalink")


def pick_source(extras: Dict[str, Any]) -> Optional[str]:
    return (
        extras.get("source")
        or extras.get("publisher")
        or extras.get("publication")
        or extras.get("source_norm")
        or extras.get("source_domain")
        or extras.get("news_outlet")
    )


DATE_KEYS = ["date", "published_at", "publishedAt", "published", "datetime", "time", "timestamp"]
EXTRA_DATE_KEYS = ["date", "published_at", "publishedAt", "published", "published_time", "pub_date", "created_at"]


def pick_raw_date(row: Dict[str, Any]) -> Any:
    for k in DATE_KEYS:
        v = row.get(k)
        if v:
            return v
    extras = parse_extras_any(row.get("extra_fields"))
    for k in EXTRA_DATE_KEYS:
        v = extras.get(k)
        if v:
            return v
    return None


def safe_date_prefix(raw_date: Any) -> str:
    s = str(raw_date or "").strip()
    return s[:10] if len(s) >= 10 else ""


def fast_parse_datetime_utc(raw_date: Any) -> Optional[datetime]:
    if raw_date is None:
        return None
    if isinstance(raw_date, (int, float)):
        try:
            x = float(raw_date)
            if x > 1e12:
                x = x / 1000.0
            return datetime.fromtimestamp(x, tz=timezone.utc)
        except Exception:
            return None
    s = str(raw_date).strip()
    if not s:
        return None
    if len(s) >= 10 and s[4] == "-" and s[7] == "-" and (len(s) == 10 or s[10] in ("T", " ")):
        try:
            if s.endswith("Z"):
                s = s[:-1] + "+00:00"
            if len(s) > 10 and s[10] == " ":
                s = s[:10] + "T" + s[11:]
            dt = datetime.fromisoformat(s if "T" in s else s[:10])
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            else:
                dt = dt.astimezone(timezone.utc)
            return dt
        except Exception:
            return None
    try:
        if s.endswith("Z"):
            s = s[:-1] + "+00:00"
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        else:
            dt = dt.astimezone(timezone.utc)
        return dt
    except Exception:
        return None


class ArticleIngestor:
    def __init__(self, db_url: Optional[str] = None):
        if db_url is None:
            db_url = build_db_url()
        logger.info(f"Connecting to database at {db_url} ...")
        self.engine = create_engine(db_url)
        self.metadata = MetaData()
        logger.info("Reflecting 'articles' table from database...")
        self.metadata.reflect(self.engine, only=["articles"])
        if "articles" not in self.metadata.tables:
            raise RuntimeError("Table 'articles' does not exist in the database.")
        self.articles: Table = self.metadata.tables["articles"]
        self.article_cols = set(self.articles.c.keys())
        logger.info(f"Successfully reflected 'articles' table. columns={sorted(self.article_cols)}")
        required = {"url", "published_at"}
        missing = [c for c in required if c not in self.article_cols]
        if missing:
            raise RuntimeError(f"'articles' table missing required columns: {missing}")

    #different from repo news_ingest.py
    #Added name: Optional[str] = None parameter + hf:// parquet loading logic
    #Used to filter relevant datasets in hugging face
    def load_dataset_any(self, streaming: bool, name: Optional[str] = None):
      token = get_hf_token_optional()
      if name:
          # Load specific subset directly from its parquet files — bypasses the 57M-row full config
          data_files = f"hf://datasets/Brianferrell787/financial-news-multisource/data/{name}/*.parquet"
          return load_dataset("parquet", data_files=data_files, split="train", streaming=streaming, token=token)
      return load_dataset(
          "Brianferrell787/financial-news-multisource",
          split="train",
          streaming=streaming,
          token=token,
      )


    def _store_articles(self, records: List[Dict[str, Any]]) -> None:
        if not records:
            return
        filtered = [{k: v for k, v in r.items() if k in self.article_cols} for r in records]
        filtered = [r for r in filtered if r]
        if not filtered:
            return
        with self.engine.begin() as conn:
            stmt = insert(self.articles).values(filtered)
            stmt = stmt.on_conflict_do_nothing(index_elements=["url"])
            conn.execute(stmt)
    #different than news_ingest.py file in the repo
    #Added name: Optional[str] = None to signature + passed it to load_dataset_any
    def ingest_all_years_one_pass(
        self,
        years: List[int],
        per_year: int = 1000,
        end_date: Optional[str] = None,
        max_scanned: int = 50_000_000,
        max_desc_chars: int = 280,
        flush_batch_size: int = 500,
        progress_every: int = 100_000,
        streaming: bool = False,
        fallback_to_non_streaming_after: int = 999_999_999,
        fallback_if_target_year_share_below: float = 0.0,
        min_rows_for_share_check: int = 2_000_000,
        name: Optional[str] = None,
    ) -> None:
        if not end_date:
            end_date = date.today().isoformat()
        years_set = set(years)
        end_year = int(end_date[:4])
        end_day = end_date[:10]
        accepted_per_year = defaultdict(int)
        year_seen = Counter()
        target_year_seen = 0

        def all_done():
            return all(accepted_per_year[y] >= per_year for y in years_set)

        label = f"[{name}]" if name else "[all]"
        logger.info(f"INGEST {label} years={sorted(years_set)} per_year={per_year} end_date={end_date}")

        ds = self.load_dataset_any(streaming=streaming, name=name)

        scanned = accepted = in_window = missing_url = bad_date = missing_date = 0
        buffer: List[Dict[str, Any]] = []
        buffer_urls = set()

        def flush():
            nonlocal buffer, buffer_urls
            if buffer:
                self._store_articles(buffer)
                buffer.clear()
                buffer_urls.clear()

        for row in ds:
            scanned += 1

            if scanned == 1:
                logger.info(f"DEBUG keys={list(row.keys())}")
                logger.info(f"DEBUG date={row.get('date')!r} published_at={row.get('published_at')!r}")

            if scanned % progress_every == 0:
                preview = " ".join([f"{y}:{accepted_per_year[y]}/{per_year}" for y in sorted(years_set)])
                logger.info(f"{label} scanned={scanned:,} accepted={accepted:,} missing_url={missing_url:,} {preview}")

            if scanned >= max_scanned:
                logger.warning(f"{label} Reached max_scanned={max_scanned:,}; stopping.")
                break

            raw_date = pick_raw_date(row)
            if not raw_date:
                missing_date += 1
                continue
            date_prefix = safe_date_prefix(raw_date)
            if not date_prefix:
                bad_date += 1
                continue

            try:
                y = int(date_prefix[:4])
                year_seen[str(y)] += 1
            except Exception:
                bad_date += 1
                continue

            if y in years_set:
                target_year_seen += 1

            if y not in years_set:
                continue
            if y == end_year and date_prefix > end_day:
                continue

            in_window += 1

            if accepted_per_year[y] >= per_year:
                if all_done():
                    logger.info(f"{label} Quotas met at scanned={scanned:,}. Stopping.")
                    break
                continue

            extras = parse_extras_any(row.get("extra_fields"))
            url = pick_url(extras)
            if not url:
                missing_url += 1
                continue
            if url in buffer_urls:
                continue

            ts = fast_parse_datetime_utc(raw_date)
            if ts is None:
                bad_date += 1
                continue

            title, desc = extract_title_and_desc(row.get("text", "") or "", max_desc_chars)
            source = pick_source(extras)
            buffer.append({"url": url, "title": title or None, "description": desc or None, "published_at": ts, "source": source or None})
            buffer_urls.add(url)
            accepted_per_year[y] += 1
            accepted += 1

            if len(buffer) >= flush_batch_size:
                flush()
            if all_done():
                logger.info(f"{label} Quotas met at scanned={scanned:,}. Stopping.")
                break

        flush()
        logger.info(f"{label} DONE. accepted={accepted:,} scanned={scanned:,} missing_url={missing_url:,}")
        short = [(y, accepted_per_year[y]) for y in sorted(years_set) if accepted_per_year[y] < per_year]
        if short:
            for y, c in short:
                logger.warning(f"  {label} SHORTFALL {y}: {c}/{per_year}")

print("ArticleIngestor defined.")

In [ ]:
import os
from datetime import date

# DATABASE_URL is already set from cell 1 via Colab secrets.
# Uncomment and replace below only if running this cell standalone:
# os.environ["DATABASE_URL"] = "put-Neon-Url-Here"

ing = ArticleIngestor()

YEARS = list(range(2020, date.today().year + 1))  # 2020 through current year
END   = str(date.today())
OPTS  = dict(years=YEARS, per_year=5000, end_date=END, streaming=False)

ing.ingest_all_years_one_pass(**OPTS, name="yahoo_finance_felixdrinkall")
ing.ingest_all_years_one_pass(**OPTS, name="benzinga_6000stocks")
ing.ingest_all_years_one_pass(**OPTS, name="reddit_finance_sp500")

'''
The following are datasets from the huggingface data folder that may
not be compatible with the current architecture/goals

These datasets may be used in the future.

#S and P is not compatible with current schema, does not return article URLS
#ing.ingest_all_years_one_pass(**OPTS, name="sp500_daily_headlines")
#Too many articles to parse through to get to relevant 2020+ data
#ing.ingest_all_years_one_pass(**OPTS, name="finsen_us_2007_2023")  # US finance with sentiment fields
#Year doesn't match with current schema
#ing.ingest_all_years_one_pass(**OPTS, name="yahoo_finance_articles") # 2025 Yahoo feed


fnspid mainly populates 2019/2020. At 8 million scans, it still has not reached 2021-2023.
ing.ingest_all_years_one_pass(**OPTS, name="fnspid_news")          # 1999–2023, finance

The reddit dataset may cause noise
reddit_finance_sp500 - Reddit finance subs (S&P-related) posts; 2008–2025; minute-level.
'''

print("All done!")